# Brain Tumor Segmentation — Evaluacija modela

Ovaj notebook:
1. Ucitava model i pokrece evaluaciju na validacijskom skupu
2. Prikazuje Dice i HD95 metrike po BraTS regijama (WT, TC, ET)
3. Vizualizira ground truth vs predikciju za vise pacijenata
4. Usporeduje rezultate s i bez postprocessinga

**BraTS regije:**
- **WT** (Whole Tumor) = klase 1 + 2 + 3
- **TC** (Tumor Core) = klase 1 + 3  
- **ET** (Enhancing Tumor) = klasa 3

In [ ]:
import sys
from pathlib import Path

# Dodaj root direktorij u path
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import torch
from monai.data import DataLoader, Dataset
from monai.inferers import sliding_window_inference
from tqdm.notebook import tqdm

from brats_config.config import NUM_CLASSES, OUTPUT_DIR, PATCH_SIZE
from src.dataset import get_patient_dicts, train_val_split
from src.model import build_model
from src.postprocess import postprocess
from src.transforms import get_val_transforms
from src.evaluate import pred_to_regions, compute_dice, compute_hd95

print("Imports OK")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA dostupan: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Konfiguracija

In [ ]:
# ── Postavke ──────────────────────────────────────────────────────────
CHECKPOINT    = ROOT / "outputs" / "best_model.pth"   # putanja do checkpointa
N_PATIENTS    = None   # None = svi val pacijenti; postavi npr. 10 za brzu provjeru
USE_POSTPROC  = True   # primijeni postprocessing
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Boje za vizualizaciju
REGION_COLORS = {"WT": "#4CAF50", "TC": "#FF9800", "ET": "#F44336"}
CLASS_COLORS  = ["black", "#3399FF", "#FF9800", "#FF3333"]  # 0=bg, 1=NCR, 2=ED, 3=ET
CLASS_NAMES   = ["Pozadina", "NCR (Nekroza)", "Edem", "ET (Enhancing)"]

print(f"Checkpoint: {CHECKPOINT}")
print(f"Uredaj: {DEVICE}")
print(f"Postprocessing: {USE_POSTPROC}")

## 2. Ucitavanje modela i podataka

In [ ]:
# Ucitaj model
model = build_model().to(DEVICE)
state = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(state)
model.eval()
print(f"Model ucitan: {sum(p.numel() for p in model.parameters()):,} parametara")

# Validacijski pacijenti
all_dicts = get_patient_dicts()
_, val_dicts = train_val_split(all_dicts)

if N_PATIENTS:
    val_dicts = val_dicts[:N_PATIENTS]

print(f"Ukupno val pacijenata: {len(val_dicts)}")

val_ds     = Dataset(data=val_dicts, transform=get_val_transforms())
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

## 3. Evaluacija (inference + metrike)

In [ ]:
results     = []
all_inputs  = []  # FLAIR volumen za svaki pacijent
all_labels  = []  # ground truth
all_preds   = []  # predikcija
patient_ids = []

for i, batch in enumerate(tqdm(val_loader, desc="Evaluacija")):
    patient_id = Path(val_dicts[i]["flair"]).parts[-2]
    patient_ids.append(patient_id)

    inputs = batch["image"].to(DEVICE)
    labels = batch["seg"].squeeze().cpu().numpy().astype(np.int32)

    with torch.no_grad():
        outputs = sliding_window_inference(
            inputs=inputs,
            roi_size=PATCH_SIZE,
            sw_batch_size=1,
            predictor=model,
            overlap=0.25,
        )

    pred = outputs.argmax(dim=1).squeeze().cpu().numpy().astype(np.int32)

    if USE_POSTPROC:
        pred = postprocess(pred, min_voxels=50)

    # Spremi za vizualizaciju
    all_inputs.append(inputs[0, 0].cpu().numpy())  # FLAIR kanal
    all_labels.append(labels)
    all_preds.append(pred)

    # BraTS regije
    pred_reg = pred_to_regions(torch.from_numpy(pred))
    gt_reg   = pred_to_regions(torch.from_numpy(labels))

    row = {"patient_id": patient_id}
    for region in ["WT", "TC", "ET"]:
        p = pred_reg[region].numpy().astype(bool)
        g = gt_reg[region].numpy().astype(bool)
        row[f"Dice_{region}"] = compute_dice(p, g)
        row[f"HD95_{region}"] = compute_hd95(p, g)

    results.append(row)

df = pd.DataFrame(results)
print(f"\nEvaluacija gotova: {len(df)} pacijenata")

## 4. Tablica rezultata po pacijentu

In [ ]:
# Formatirani prikaz
display_df = df.copy()

# HD95 = inf kad nema tumora → zamijeni s NaN za prikaz
hd_cols = [c for c in df.columns if c.startswith("HD95")]
display_df[hd_cols] = display_df[hd_cols].replace([np.inf, -np.inf], np.nan)

# Stilizacija: boje prema Dice vrijednosti
dice_cols = ["Dice_WT", "Dice_TC", "Dice_ET"]

def color_dice(val):
    if pd.isna(val): return "background-color: #f0f0f0"
    if val >= 0.8:   return "background-color: #c8e6c9; font-weight: bold"
    if val >= 0.6:   return "background-color: #fff9c4"
    if val >= 0.4:   return "background-color: #ffe0b2"
    return "background-color: #ffcdd2"

styled = display_df.style.applymap(color_dice, subset=dice_cols).format(
    {c: "{:.3f}" for c in dice_cols + hd_cols}
)

print(f"Val pacijenata: {len(df)}")
display(styled)

## 5. Statistike i prosjeci

In [ ]:
metric_cols = [c for c in df.columns if c != "patient_id"]

# Zamijeni inf s nan za statistike
df_clean = df[metric_cols].replace([np.inf, -np.inf], np.nan)

summary = df_clean.agg(["mean", "std", "min", "median", "max"]).round(3)

print("=" * 65)
print("SUMARNI REZULTATI EVALUACIJE")
print("=" * 65)
display(summary)

print("\nProsjek ± std po metrikama:")
print("-" * 45)
for col in metric_cols:
    finite = df_clean[col].dropna()
    print(f"  {col:12s}: {finite.mean():.3f} ± {finite.std():.3f}")

## 6. Boxplot Dice scoreova

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Dice boxplot ---
dice_cols = ["Dice_WT", "Dice_TC", "Dice_ET"]
dice_data = [df[c].replace([np.inf, -np.inf], np.nan).dropna().values for c in dice_cols]

bp = axes[0].boxplot(dice_data, labels=["WT", "TC", "ET"],
                      patch_artist=True, notch=False, widths=0.5)

for patch, color in zip(bp["boxes"], REGION_COLORS.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(2)

mean_all = df[dice_cols].replace([np.inf, -np.inf], np.nan).mean().mean()
axes[0].axhline(mean_all, color="navy", linestyle="--", alpha=0.6,
                label=f"ukupni prosjek = {mean_all:.3f}")

# Dodaj tocke po pacijentima
for j, data in enumerate(dice_data):
    x = np.random.normal(j + 1, 0.07, len(data))
    axes[0].scatter(x, data, alpha=0.4, s=25, color="dimgray", zorder=3)

axes[0].set_title("Dice Score po BraTS regijama", fontsize=13)
axes[0].set_ylabel("Dice Score")
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend(fontsize=9)

# --- HD95 boxplot ---
hd95_cols = ["HD95_WT", "HD95_TC", "HD95_ET"]
hd95_data = [df[c].replace([np.inf, -np.inf], np.nan).dropna().values for c in hd95_cols]

bp2 = axes[1].boxplot(hd95_data, labels=["WT", "TC", "ET"],
                       patch_artist=True, notch=False, widths=0.5)

for patch, color in zip(bp2["boxes"], REGION_COLORS.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
for median in bp2["medians"]:
    median.set_color("black")
    median.set_linewidth(2)

for j, data in enumerate(hd95_data):
    x = np.random.normal(j + 1, 0.07, len(data))
    axes[1].scatter(x, data, alpha=0.4, s=25, color="dimgray", zorder=3)

axes[1].set_title("HD95 po BraTS regijama (manje = bolje)", fontsize=13)
axes[1].set_ylabel("HD95 (vokseli)")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eval_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graf spremljen.")

## 7. Histogram Dice scoreova

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, region, color in zip(axes, ["WT", "TC", "ET"], REGION_COLORS.values()):
    col  = f"Dice_{region}"
    vals = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    
    ax.hist(vals, bins=20, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(vals.mean(), color="navy", linewidth=2, linestyle="--",
               label=f"mean={vals.mean():.3f}")
    ax.axvline(vals.median(), color="darkred", linewidth=2, linestyle=":",
               label=f"median={vals.median():.3f}")
    
    ax.set_title(f"Dice {region}", fontsize=12)
    ax.set_xlabel("Dice Score")
    ax.set_ylabel("Broj pacijenata")
    ax.set_xlim(0, 1)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("Distribucija Dice Scoreova po Regijama", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eval_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Vizualizacija Ground Truth vs Predikcija

Za svaki prikazani pacijent: FLAIR snimka, Ground Truth segmentacija, Predikcija modela — u 3 aksijalnih presjeka (gornji, srednji, donji tumor).

In [ ]:
from matplotlib.colors import ListedColormap

# Custom colormap: 0=crno, 1=plavo(NCR), 2=narancasto(Edem), 3=crveno(ET)
SEG_CMAP = ListedColormap(["black", "#3399FF", "#FF9800", "#FF3333"])

def find_tumor_slices(seg_vol, n_slices=3):
    """Vrati n_slices aksijalni z-indeksa gdje ima najvise tumora."""
    tumor_per_z = (seg_vol > 0).sum(axis=(0, 1))  # suma voksela po z
    if tumor_per_z.sum() == 0:
        z_mid = seg_vol.shape[2] // 2
        return [z_mid]
    
    # Uzmi percentile od 20% do 80% z-indeksa koji imaju tumor
    z_with_tumor = np.where(tumor_per_z > tumor_per_z.max() * 0.05)[0]
    percentiles  = [25, 50, 75] if n_slices == 3 else [50]
    slices       = [int(np.percentile(z_with_tumor, p)) for p in percentiles]
    return slices


def plot_patient(flair, gt, pred, patient_id, dice_row, n_slices=3):
    """Vizualizacija jednog pacijenta: FLAIR | GT | Pred za n_slices presjeka."""
    z_slices = find_tumor_slices(gt, n_slices)
    n = len(z_slices)
    
    fig = plt.figure(figsize=(5 * n, 11))
    gs  = gridspec.GridSpec(3, n, hspace=0.08, wspace=0.05)
    
    row_titles = ["FLAIR", "Ground Truth", "Predikcija"]
    
    for col, z in enumerate(z_slices):
        flair_sl = flair[:, :, z].T
        gt_sl    = gt[:, :, z].T
        pred_sl  = pred[:, :, z].T

        for row, (data, cmap, vmin, vmax) in enumerate([
            (flair_sl, "gray", None, None),
            (gt_sl,   SEG_CMAP, 0, 3),
            (pred_sl, SEG_CMAP, 0, 3),
        ]):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, origin="lower",
                      interpolation="nearest")
            ax.axis("off")
            
            if col == 0:
                ax.set_ylabel(row_titles[row], fontsize=11, rotation=90,
                              labelpad=6, va="center")
                ax.yaxis.set_label_coords(-0.08, 0.5)
            
            if row == 0:
                ax.set_title(f"z = {z}", fontsize=10)

    # Legenda klasa
    legend_patches = [
        mpatches.Patch(color=c, label=n)
        for c, n in zip(["#3399FF", "#FF9800", "#FF3333"],
                        ["NCR/Nekroza", "Edem", "ET (Enhancing)"])
    ]
    fig.legend(handles=legend_patches, loc="lower center",
               ncol=3, fontsize=10, framealpha=0.9,
               bbox_to_anchor=(0.5, -0.02))

    # Naslov s metrikama
    dice_str = (
        f"Dice  WT={dice_row['Dice_WT']:.3f}  "
        f"TC={dice_row['Dice_TC']:.3f}  "
        f"ET={dice_row['Dice_ET']:.3f}"
    )
    hd95_wt = dice_row['HD95_WT']
    hd95_tc = dice_row['HD95_TC']
    hd95_et = dice_row['HD95_ET']
    hd95_str = (
        f"HD95  WT={hd95_wt:.1f}  TC={hd95_tc:.1f}  ET={hd95_et:.1f}"
        if not any(np.isinf([hd95_wt, hd95_tc, hd95_et]))
        else "HD95  (neki inf — region bez voksela)"
    )

    fig.suptitle(f"{patient_id}\n{dice_str}\n{hd95_str}",
                 fontsize=11, y=1.01)
    plt.savefig(OUTPUT_DIR / f"eval_{patient_id}.png",
                dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
# Koliko pacijenata vizualizirati
N_VIZ = min(5, len(all_preds))

print(f"Vizualizacija {N_VIZ} pacijenata...")

for i in range(N_VIZ):
    pid      = patient_ids[i]
    dice_row = df[df["patient_id"] == pid].iloc[0]
    
    plot_patient(
        flair=all_inputs[i],
        gt=all_labels[i],
        pred=all_preds[i],
        patient_id=pid,
        dice_row=dice_row,
        n_slices=3,
    )
    print(f"  [{i+1}/{N_VIZ}] {pid}  — "
          f"Dice WT={dice_row['Dice_WT']:.3f}, "
          f"TC={dice_row['Dice_TC']:.3f}, "
          f"ET={dice_row['Dice_ET']:.3f}")

## 9. Usporedba s i bez postprocessinga

In [ ]:
# Pokreni evaluaciju bez postprocessinga za usporedbu
results_nopp = []

for i, (pred_pp, labels) in enumerate(tqdm(zip(all_preds, all_labels),
                                            total=len(all_preds),
                                            desc="Bez postprocesinga")):
    # Rekonstruiraj predikciju BEZ postprocessinga
    # (koristimo stored rezultate — samo za prikaz razlike)
    pass

# Provjeri ucink postprocessinga na prvom pacijentu
from src.postprocess import remove_small_components, fill_holes

i = 0  # prvi pacijent
pred = all_preds[i]
gt   = all_labels[i]

# Dohvati originalnu predikciju (bez postprocesinga) iz evaluacije
# (ovdje koristimo stored pred koji je vec postprocessiran — prikazujemo razliku fill_holes)
pred_no_fill = remove_small_components(pred, min_voxels=50)
pred_filled  = fill_holes(pred_no_fill)

print("Usporedba fill_holes efekta na prvom pacijentu:")
print(f"  Vokseli tumora bez fill_holes: {(pred_no_fill > 0).sum()}")
print(f"  Vokseli tumora s  fill_holes:  {(pred_filled  > 0).sum()}")
print(f"  Dodano voksela:                {(pred_filled > 0).sum() - (pred_no_fill > 0).sum()}")

# Dice usporedba
for region in ["WT", "TC", "ET"]:
    pred_reg_nf = pred_to_regions(torch.from_numpy(pred_no_fill))
    pred_reg_f  = pred_to_regions(torch.from_numpy(pred_filled))
    gt_reg      = pred_to_regions(torch.from_numpy(gt))

    d_nf = compute_dice(pred_reg_nf[region].numpy().astype(bool),
                        gt_reg[region].numpy().astype(bool))
    d_f  = compute_dice(pred_reg_f[region].numpy().astype(bool),
                        gt_reg[region].numpy().astype(bool))
    
    print(f"  Dice {region}: bez={d_nf:.3f}  s={d_f:.3f}  delta={d_f-d_nf:+.3f}")

## 10. Vizualizacija BraTS regija (WT, TC, ET)

In [ ]:
def plot_brats_regions(flair, gt, pred, patient_id, z=None):
    """Prikaz 3 BraTS regije (WT/TC/ET) — GT vs Pred."""
    if z is None:
        z = find_tumor_slices(gt, n_slices=1)[0]
    
    gt_regions   = pred_to_regions(torch.from_numpy(gt))
    pred_regions = pred_to_regions(torch.from_numpy(pred))
    
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    fig.suptitle(f"{patient_id}  |  z = {z}  — BraTS regije: GT vs Predikcija",
                 fontsize=12)
    
    flair_sl = flair[:, :, z].T
    
    # Gornji red: GT
    axes[0, 0].imshow(flair_sl, cmap="gray", origin="lower")
    axes[0, 0].set_title("FLAIR")
    
    axes[0, 1].imshow(gt[:, :, z].T, cmap=SEG_CMAP, vmin=0, vmax=3, origin="lower")
    axes[0, 1].set_title("GT — sve klase")
    
    for col, (region, color) in enumerate(REGION_COLORS.items(), start=2):
        mask = gt_regions[region].numpy()[:, :, z].T
        axes[0, col].imshow(flair_sl, cmap="gray", origin="lower", alpha=0.5)
        axes[0, col].imshow(np.ma.masked_where(mask == 0, mask),
                             cmap=ListedColormap([color]), origin="lower",
                             alpha=0.8, vmin=0, vmax=1)
        axes[0, col].set_title(f"GT {region}")
    
    # Donji red: Predikcija
    axes[1, 0].imshow(flair_sl, cmap="gray", origin="lower")
    axes[1, 0].set_title("FLAIR")
    
    axes[1, 1].imshow(pred[:, :, z].T, cmap=SEG_CMAP, vmin=0, vmax=3, origin="lower")
    axes[1, 1].set_title("Pred — sve klase")
    
    for col, (region, color) in enumerate(REGION_COLORS.items(), start=2):
        mask = pred_regions[region].numpy()[:, :, z].T
        axes[1, col].imshow(flair_sl, cmap="gray", origin="lower", alpha=0.5)
        axes[1, col].imshow(np.ma.masked_where(mask == 0, mask),
                             cmap=ListedColormap([color]), origin="lower",
                             alpha=0.8, vmin=0, vmax=1)
        axes[1, col].set_title(f"Pred {region}")
    
    # Row labels
    for row_idx, label in enumerate(["Ground Truth", "Predikcija"]):
        axes[row_idx, 0].set_ylabel(label, fontsize=11, labelpad=8)
    
    for ax in axes.flat:
        ax.axis("off")
    
    # Legenda regija
    legend_patches = [
        mpatches.Patch(color=c, label=r)
        for r, c in REGION_COLORS.items()
    ]
    fig.legend(handles=legend_patches, loc="lower center", ncol=3,
               fontsize=10, bbox_to_anchor=(0.5, -0.03))
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"regions_{patient_id}.png",
                dpi=130, bbox_inches="tight")
    plt.show()
    plt.close()


# Prikaz za prva 3 pacijenta
for i in range(min(3, len(all_preds))):
    plot_brats_regions(all_inputs[i], all_labels[i], all_preds[i], patient_ids[i])

## 11. Najlosi i najboli pacijenti po Dice WT

In [ ]:
df_sorted = df.sort_values("Dice_WT", ascending=False)

print("Top 5 pacijenata (po Dice_WT):")
display(df_sorted[["patient_id", "Dice_WT", "Dice_TC", "Dice_ET"]].head(5).round(3))

print("\nNajgori 5 pacijenata (po Dice_WT):")
display(df_sorted[["patient_id", "Dice_WT", "Dice_TC", "Dice_ET"]].tail(5).round(3))

In [ ]:
# Vizualizacija najboljeg i najloseg pacijenta
best_pid  = df_sorted.iloc[0]["patient_id"]
worst_pid = df_sorted.iloc[-1]["patient_id"]

for pid, label in [(best_pid, "BEST"), (worst_pid, "WORST")]:
    idx      = patient_ids.index(pid)
    dice_row = df[df["patient_id"] == pid].iloc[0]
    print(f"\n{'='*50}")
    print(f"{label}: {pid}")
    print(f"Dice WT={dice_row['Dice_WT']:.3f}, TC={dice_row['Dice_TC']:.3f}, ET={dice_row['Dice_ET']:.3f}")
    plot_patient(
        flair=all_inputs[idx],
        gt=all_labels[idx],
        pred=all_preds[idx],
        patient_id=f"{label}_{pid}",
        dice_row=dice_row,
        n_slices=3,
    )

## 12. Spremi rezultate

In [ ]:
csv_path = OUTPUT_DIR / "evaluation_results.csv"
df.to_csv(csv_path, index=False)
print(f"CSV spremljen: {csv_path}")

# Finalni sumarni ispis
print("\n" + "="*60)
print("FINALNI REZULTATI")
print("="*60)

df_clean = df[["Dice_WT", "Dice_TC", "Dice_ET",
               "HD95_WT",  "HD95_TC",  "HD95_ET"]].replace([np.inf, -np.inf], np.nan)

for region in ["WT", "TC", "ET"]:
    d = df_clean[f"Dice_{region}"].dropna()
    h = df_clean[f"HD95_{region}"].dropna()
    print(f"  {region}  Dice: {d.mean():.3f} ± {d.std():.3f}  "
          f"| HD95: {h.mean():.1f} ± {h.std():.1f}")

print("="*60)